# 🧪 Chemora Extract — Colab GPU runner

Runs the **full pipeline** (OpenChemIE: molecules **and** reactions) on a free Colab GPU.
This is the honest "it actually works" path — Colab has the GPU and open network the
models need to download weights and run at usable speed.

**Before running:** `Runtime → Change runtime type → GPU` (T4 is fine).

Two ways to use it:
1. **Inline** (cells 1–5): upload a PDF, see extracted SMILES + drawn structures right here. Most reliable for a demo.
2. **Web app** (cell 6, optional): launch the Streamlit UI with a public link.


## 1 · Check the GPU


In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())

## 2 · Install dependencies
First run takes a few minutes (downloads packages). If an install conflicts,
the error names the offending package — pin a different version and re-run just this cell.


In [ ]:
# System: poppler is REQUIRED (OpenChemIE uses pdf2image -> poppler).
!apt-get -qq install -y poppler-utils
# Core
!pip install -q pymupdf pillow 'numpy<2.1' rdkit streamlit streamlit-ketcher
# OpenChemIE (molecules + reactions). Pulls MolScribe + RxnScribe + weights on first use.
!pip install -q openchemie
print('\n✅ installs done')

## 3 · Get the project code
Easiest: upload the `chemora-extract.zip` you already have. (Or set a GitHub URL and clone instead.)


In [ ]:
import os, zipfile

USE_GITHUB = False            # set True and fill REPO_URL to clone instead of upload
REPO_URL   = ''              # e.g. 'https://github.com/vvvvvivekkk/chemora-extract.git'

if USE_GITHUB and REPO_URL:
    !git clone $REPO_URL chemora-extract
else:
    from google.colab import files
    print('Upload chemora-extract.zip …')
    up = files.upload()                       # pick the zip
    zname = next(iter(up))
    with zipfile.ZipFile(zname) as z:
        z.extractall('.')

# locate the folder that contains app.py
root = None
for d, _, fs in os.walk('.'):
    if 'app.py' in fs and os.path.isdir(os.path.join(d, 'pipeline')):
        root = d; break
assert root, 'Could not find the project (app.py + pipeline/). Check the upload.'
os.chdir(root)
print('Project root:', os.getcwd())

## 4 · Upload a chemistry PDF and run the pipeline
Uses the real `pipeline.orchestrator` — the same code the web app uses.
First run loads model weights (slow once, cached after).


In [ ]:
import os
os.environ['CHEMORA_ENGINE'] = 'openchemie'   # molecules + reactions
os.environ['CHEMORA_MAX_PAGES'] = '5'         # keep the demo quick

from google.colab import files
print('Upload a chemistry PDF …')
up = files.upload()
pdf_name = next(iter(up))
pdf_bytes = up[pdf_name]

from pipeline.orchestrator import process_pdf
result = process_pdf(pdf_bytes, progress_callback=lambda f,m: print(f'[{int(f*100):3d}%] {m}'))
print('\nMolecules:', result.total_molecules, '| Reactions:', result.total_reactions)

## 5 · Show the results (structures drawn with RDKit)


In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

for page in result.pages:
    if not page.molecules and not page.reactions:
        continue
    print(f'\n===== Page {page.index + 1} =====')
    mols, legends = [], []
    for m in page.molecules:
        smi = m.canonical_smiles or m.raw_smiles
        mol = Chem.MolFromSmiles(smi) if m.is_valid else None
        if mol is not None:
            mols.append(mol); legends.append(smi)
        else:
            print('  ⚠️ unparseable:', m.raw_smiles)
    if mols:
        display(Draw.MolsToGridImage(mols, legends=legends, molsPerRow=3, subImgSize=(260,220)))
    for r in page.reactions:
        print('  ⚗️ reaction:', r.as_reaction_smiles())
        if r.conditions:
            print('     conditions:', '; '.join(r.conditions))

## 6 · (Optional) Launch the full Streamlit web app with a public link
Runs the real UI (side-by-side page + editable Ketcher canvas). The `cloudflared`
line prints a public URL you can open (and screen-share).


In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import subprocess, time, re, threading
os.environ['CHEMORA_ENGINE'] = 'openchemie'
proc = subprocess.Popen(['streamlit','run','app.py','--server.port','8501','--server.headless','true'])
time.sleep(6)
tun = subprocess.Popen(['./cloudflared','tunnel','--url','http://localhost:8501'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
def _print():
    for line in tun.stdout:
        if 'trycloudflare.com' in line:
            print('\n🌐 Open your app at:', re.search(r'https://[^ ]+trycloudflare.com', line).group(0))
threading.Thread(target=_print, daemon=True).start()
print('Starting… the public URL will appear in a few seconds.')